In [1]:
import os
import re

import numpy as np
import pandas as pd


# ============================================================
# Analysis configuration
# ============================================================
CURRENT_DENSITIES = [
    50,
    100,
    200,
    300,
    400,
    500,
]

PREDICTION_DIRECTORY = "results"
OUTPUT_DIRECTORY = "metal"

PREDICTION_COLUMN = "Predicted_Overpotential"
LOW_OVERPOTENTIAL_FRACTION = 0.1

os.makedirs(
    OUTPUT_DIRECTORY,
    exist_ok=True,
)

In [2]:
# ============================================================
# Parse a chemical formula
# ============================================================
def parse_formula_elements_and_fractions(formula):
    """
    Extract elemental symbols and stoichiometric coefficients
    from a chemical formula.
    """
    tokens = re.findall(
        r"([A-Z][a-z]*)(\d*\.?\d*)",
        str(formula),
    )

    parsed_formula = []

    for element, coefficient in tokens:
        value = (
            float(coefficient)
            if coefficient
            else 1.0
        )

        parsed_formula.append(
            (element, value)
        )

    return parsed_formula


# ============================================================
# Extract metal elements
# ============================================================
def extract_metal_information(formula):
    """
    Extract the metal elements and their stoichiometric
    coefficients.

    The final element in the formula is treated as the
    nonmetal element.
    """
    parsed_formula = (
        parse_formula_elements_and_fractions(
            formula
        )
    )

    if not parsed_formula:
        return [], []

    metal_entries = parsed_formula[:-1]

    metals = [
        element
        for element, _ in metal_entries
    ]

    metal_fractions = [
        fraction
        for _, fraction in metal_entries
    ]

    return metals, metal_fractions


# ============================================================
# Identify the main and auxiliary metals
# ============================================================
def identify_metal_roles(formula):
    """
    Define the metal with the highest stoichiometric
    coefficient as the main metal and the remaining metals
    as auxiliary metals.
    """
    metals, metal_fractions = (
        extract_metal_information(formula)
    )

    if not metals:
        return [], None, []

    main_index = int(
        np.argmax(metal_fractions)
    )

    main_metal = metals[main_index]

    auxiliary_metals = [
        metal
        for index, metal in enumerate(metals)
        if index != main_index
    ]

    return (
        metals,
        main_metal,
        auxiliary_metals,
    )

In [3]:
# ============================================================
# Analyze auxiliary-metal effects at one current density
# ============================================================
def analyze_current_density(current_density):
    """
    Calculate auxiliary-metal enrichment within groups of
    candidates sharing the same main metal.
    """
    input_path = os.path.join(
        PREDICTION_DIRECTORY,
        (
            "five_metal_compounds_predictions_"
            f"{current_density}.csv"
        ),
    )

    data = pd.read_csv(input_path)

    required_columns = {
        "compound",
        PREDICTION_COLUMN,
    }

    missing_columns = required_columns.difference(
        data.columns
    )

    if missing_columns:
        raise KeyError(
            f"Missing columns in {input_path}: "
            f"{sorted(missing_columns)}"
        )


    # --------------------------------------------------------
    # Identify the main and auxiliary metals
    # --------------------------------------------------------
    metal_lists = []
    main_metals = []
    auxiliary_metal_lists = []

    for formula in data["compound"].astype(str):
        (
            metals,
            main_metal,
            auxiliary_metals,
        ) = identify_metal_roles(formula)

        metal_lists.append(metals)
        main_metals.append(main_metal)
        auxiliary_metal_lists.append(
            auxiliary_metals
        )

    data["metal_list"] = metal_lists
    data["main_metal"] = main_metals
    data["dopant_list"] = auxiliary_metal_lists


    # --------------------------------------------------------
    # Identify the lowest 5% predicted overpotentials
    # --------------------------------------------------------
    overpotential_threshold = data[
        PREDICTION_COLUMN
    ].quantile(
        LOW_OVERPOTENTIAL_FRACTION
    )

    data["is_top"] = (
        data[PREDICTION_COLUMN]
        <= overpotential_threshold
    )


    # --------------------------------------------------------
    # Expand the auxiliary-metal lists
    # --------------------------------------------------------
    expanded_data = (
        data[
            [
                "compound",
                "main_metal",
                "dopant_list",
                PREDICTION_COLUMN,
                "is_top",
            ]
        ]
        .explode("dopant_list")
        .rename(
            columns={
                "dopant_list": "dopant"
            }
        )
        .dropna(
            subset=["dopant"]
        )
    )


    # --------------------------------------------------------
    # Calculate auxiliary-metal statistics within each
    # main-metal group
    # --------------------------------------------------------
    dopant_by_main = (
        expanded_data
        .groupby(
            [
                "main_metal",
                "dopant",
            ]
        )
        .agg(
            n=(
                "compound",
                "count",
            ),
            mean_pred=(
                PREDICTION_COLUMN,
                "mean",
            ),
            median_pred=(
                PREDICTION_COLUMN,
                "median",
            ),
            top_rate=(
                "is_top",
                "mean",
            ),
        )
        .reset_index()
    )


    # --------------------------------------------------------
    # Calculate the baseline for each main-metal group
    # --------------------------------------------------------
    main_baseline = (
        data
        .groupby("main_metal")
        .agg(
            baseline_mean=(
                PREDICTION_COLUMN,
                "mean",
            ),
            baseline_median=(
                PREDICTION_COLUMN,
                "median",
            ),
            baseline_top_rate=(
                "is_top",
                "mean",
            ),
        )
        .reset_index()
    )

    dopant_by_main = dopant_by_main.merge(
        main_baseline,
        on="main_metal",
        how="left",
    )


    # --------------------------------------------------------
    # Calculate the conditional auxiliary-metal effects
    # --------------------------------------------------------
    dopant_by_main[
        "delta_median_vs_main"
    ] = (
        dopant_by_main["median_pred"]
        - dopant_by_main["baseline_median"]
    )

    dopant_by_main[
        "top_rate_enrichment_vs_main"
    ] = (
        dopant_by_main["top_rate"]
        / dopant_by_main[
            "baseline_top_rate"
        ]
    )

    dopant_by_main = (
        dopant_by_main
        .sort_values(
            "top_rate_enrichment_vs_main",
            ascending=False,
        )
        .reset_index(drop=True)
    )


    # --------------------------------------------------------
    # Summarize each auxiliary metal across main-metal groups
    # --------------------------------------------------------
    dopant_summary = (
        dopant_by_main
        .groupby("dopant")
        .agg(
            n_main_metal=(
                "main_metal",
                "nunique",
            ),
            total_count=(
                "n",
                "sum",
            ),
            mean_delta_median_vs_main=(
                "delta_median_vs_main",
                "mean",
            ),
            median_delta_median_vs_main=(
                "delta_median_vs_main",
                "median",
            ),
            mean_top_rate_enrichment=(
                "top_rate_enrichment_vs_main",
                "mean",
            ),
            median_top_rate_enrichment=(
                "top_rate_enrichment_vs_main",
                "median",
            ),
            favorable_main_fraction=(
                "delta_median_vs_main",
                lambda values: (
                    values < 0
                ).mean(),
            ),
        )
        .reset_index()
    )

    dopant_summary = (
        dopant_summary
        .sort_values(
            "median_top_rate_enrichment",
            ascending=False,
        )
        .reset_index(drop=True)
    )

    dopant_summary[
        "CurrentDensity"
    ] = current_density


    # --------------------------------------------------------
    # Save the detailed and summarized results
    # --------------------------------------------------------
    detailed_output_path = os.path.join(
        OUTPUT_DIRECTORY,
        (
            "dopant_effect_conditioned_by_"
            f"main_metal_{current_density}.csv"
        ),
    )

    summary_output_path = os.path.join(
        OUTPUT_DIRECTORY,
        (
            "dopant_effect_summary_conditioned_by_"
            f"main_metal_{current_density}.csv"
        ),
    )

    dopant_by_main.to_csv(
        detailed_output_path,
        index=False,
    )

    dopant_summary.to_csv(
        summary_output_path,
        index=False,
    )

    print(
        f"{current_density} mA g-1 | "
        f"threshold: {overpotential_threshold:.6f} V | "
        f"candidates: {len(data)}"
    )

    return (
        data,
        dopant_by_main,
        dopant_summary,
    )

In [4]:
# ============================================================
# Run the auxiliary-metal analysis
# ============================================================
prediction_results = {}
conditional_results = {}
summary_results = {}

for current_density in CURRENT_DENSITIES:
    (
        prediction_data,
        conditional_data,
        summary_data,
    ) = analyze_current_density(
        current_density
    )

    prediction_results[
        current_density
    ] = prediction_data

    conditional_results[
        current_density
    ] = conditional_data

    summary_results[
        current_density
    ] = summary_data


# ============================================================
# Combine all current-density summaries
# ============================================================
all_dopant_summary = pd.concat(
    [
        summary_results[current_density]
        for current_density
        in CURRENT_DENSITIES
    ],
    ignore_index=True,
)

all_dopant_summary.to_csv(
    os.path.join(
        OUTPUT_DIRECTORY,
        "dopant_effect_summary_all_current_densities.csv",
    ),
    index=False,
)

display(
    all_dopant_summary[
        [
            "CurrentDensity",
            "dopant",
            "median_top_rate_enrichment",
            "median_delta_median_vs_main",
            "favorable_main_fraction",
        ]
    ]
)

50 mA g-1 | threshold: 0.802881 V | candidates: 415800
100 mA g-1 | threshold: 0.885584 V | candidates: 415800
200 mA g-1 | threshold: 1.034674 V | candidates: 415800
300 mA g-1 | threshold: 1.144378 V | candidates: 415800
400 mA g-1 | threshold: 1.221048 V | candidates: 415800
500 mA g-1 | threshold: 1.275150 V | candidates: 415800


,CurrentDensity,dopant,median_top_rate_enrichment,median_delta_median_vs_main,favorable_main_fraction
0,50,Pd,1.306706,-0.023007,0.625000
1,50,Mo,1.297393,-0.034317,0.666667
2,50,In,1.180684,-0.009061,0.708333
3,50,Ag,1.153036,-0.030036,0.708333
4,50,Ru,1.127073,-0.019222,0.666667
...,...,...,...,...,...
61,500,In,0.979117,-0.005303,0.541667
62,500,Co,0.941551,0.007442,0.333333
63,500,Zn,0.817439,0.007433,0.416667
64,500,Fe,0.786811,0.027537,0.166667


In [5]:
# ============================================================
# Determine favorable associations at each current density
# ============================================================
all_dopant_summary[
    "is_favorable"
] = (
    (
        all_dopant_summary[
            "median_delta_median_vs_main"
        ] < 0
    )
    & (
        all_dopant_summary[
            "median_top_rate_enrichment"
        ] > 1
    )
    & (
        all_dopant_summary[
            "favorable_main_fraction"
        ] > 0.5
    )
)


# ============================================================
# Summarize each auxiliary metal across current densities
# ============================================================
robust_dopant_summary = (
    all_dopant_summary
    .groupby("dopant")
    .agg(
        n_current_density=(
            "CurrentDensity",
            "nunique",
        ),
        n_favorable_current_density=(
            "is_favorable",
            "sum",
        ),
        mean_delta_median_vs_main=(
            "median_delta_median_vs_main",
            "mean",
        ),
        median_delta_median_vs_main=(
            "median_delta_median_vs_main",
            "median",
        ),
        max_delta_median_vs_main=(
            "median_delta_median_vs_main",
            "max",
        ),
        mean_top_rate_enrichment=(
            "median_top_rate_enrichment",
            "mean",
        ),
        median_top_rate_enrichment=(
            "median_top_rate_enrichment",
            "median",
        ),
        min_top_rate_enrichment=(
            "median_top_rate_enrichment",
            "min",
        ),
        mean_favorable_main_fraction=(
            "favorable_main_fraction",
            "mean",
        ),
        median_favorable_main_fraction=(
            "favorable_main_fraction",
            "median",
        ),
        min_favorable_main_fraction=(
            "favorable_main_fraction",
            "min",
        ),
        total_count=(
            "total_count",
            "sum",
        ),
    )
    .reset_index()
)


# ============================================================
# Identify metals favorable at all six current densities
# ============================================================
robust_dopant_summary[
    "favorable_all_current_densities"
] = (
    robust_dopant_summary[
        "n_favorable_current_density"
    ]
    == len(CURRENT_DENSITIES)
)


# ============================================================
# Rank auxiliary metals by mean enrichment ratio
# ============================================================
robust_dopant_summary = (
    robust_dopant_summary
    .sort_values(
        [
            "favorable_all_current_densities",
            "n_favorable_current_density",
            "mean_top_rate_enrichment",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)


# ============================================================
# Extract metals favorable at all current densities
# ============================================================
favorable_dopants = (
    robust_dopant_summary.loc[
        robust_dopant_summary[
            "favorable_all_current_densities"
        ]
    ]
    .sort_values(
        "mean_top_rate_enrichment",
        ascending=False,
    )
    .reset_index(drop=True)
)


# ============================================================
# Save the final results
# ============================================================
robust_dopant_summary.to_csv(
    os.path.join(
        OUTPUT_DIRECTORY,
        "robust_favorable_dopants_across_current_densities.csv",
    ),
    index=False,
)

favorable_dopants.to_csv(
    os.path.join(
        OUTPUT_DIRECTORY,
        "favorable_dopants_all_six_current_densities.csv",
    ),
    index=False,
)


# ============================================================
# Display the final enrichment ranking
# ============================================================
print(
    "Auxiliary metals favorable at all six current "
    "densities, ranked by mean enrichment ratio:"
)

display_columns = [
    "dopant",
    "mean_top_rate_enrichment",
    "median_top_rate_enrichment",
    "n_favorable_current_density",
    "mean_delta_median_vs_main",
    "mean_favorable_main_fraction",
]

display(
    favorable_dopants[
        display_columns
    ]
)

Auxiliary metals favorable at all six current densities, ranked by mean enrichment ratio:


,dopant,mean_top_rate_enrichment,median_top_rate_enrichment,n_favorable_current_density,mean_delta_median_vs_main,mean_favorable_main_fraction
0,Pd,1.252287,1.256489,6,-0.022404,0.694444
1,Mo,1.186516,1.162910,6,-0.018636,0.618056
2,Ag,1.139064,1.147120,6,-0.021735,0.701389
3,Ru,1.110691,1.113537,6,-0.015413,0.604167
